In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.messages import BaseMessage, HumanMessage
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver

c:\Users\rohit\Desktop\LangGraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
load_dotenv()

True

In [5]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation"
)
model = ChatHuggingFace(llm=llm)

In [6]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [9]:
def chat_node(state: ChatState):

    # take the user's query from state
    messages = state['messages']
    
    # send to llm
    response = model.invoke(messages)

    # store the response in state
    return {'messages': [response]}

In [1]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

# add nodes
graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile(checkpointer=checkpointer)

chatbot

NameError: name 'MemorySaver' is not defined

In [12]:
initial_state = {
    'messages': [HumanMessage(content="what is the capital of india")]
}

chatbot.invoke(initial_state)['messages'][-1].content

'The capital of India is New Delhi. It is located in the Delhi metropolitan area and is one of the 11 districts in the National Capital Territory of Delhi. New Delhi was planned and constructed to serve as the capital of India in the early 20th century, replacing Kolkata (formerly known as Calcutta) as the capital.'

In [16]:
thread_id = '1'

while True:

    user_message = input("Type here")

    print("User : ", user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    config = {'configurable': {'thread_id': thread_id}}
    response = chatbot.invoke({'messages': [HumanMessage(content = user_message)]}, config=config)

    print("AI : ", response['messages'][-1].content)

User :  hi my name is rohit
AI :  Hello Rohit! Nice to meet you. How can I assist you today?
User :  what is my name
AI :  Your name is Rohit.
User :  can you add 10 to 110
AI :  Sure! When you add 10 to 110, the result is 120.
User :  can you multiply result with 2
AI :  Certainly! If you multiply the result of 120 by 2, you get 240.
User :  exit


In [17]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='hi my name is rohit', additional_kwargs={}, response_metadata={}, id='8ad220b9-1a7a-43a9-8b70-449cfc157795'), AIMessage(content='Hello Rohit! Nice to meet you. How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 35, 'total_tokens': 52}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da971-38a2-7563-9084-0b328414f2ed-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 35, 'output_tokens': 17, 'total_tokens': 52}), HumanMessage(content='what is my name', additional_kwargs={}, response_metadata={}, id='080865ec-6608-41d5-b46d-4e4072e8a28a'), AIMessage(content='Your name is Rohit.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 65, 'total_tokens': 72}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fin